In [0]:
#%run ./env ----- A decommenter pour lancer le notebook separement
#%run ./python_libraries ----- A decommenter pour lancer le notebook separement

## Definir la source en fonction de l'environnement
TODO: adapter `source_catalog` si ce nouveau projet pointe vers un catalog different de l'ancien.

## Chargement des tables sources
Pur chargement, aucune transformation ici (coherent avec le pattern de l'ancien projet).

In [0]:
batches = spark.table(f"{source_catalog}.batches")
plants_production_lines = spark.table(f"{source_catalog}.plants_production_lines")
parameters_variables = spark.table(f"{source_catalog}.parameters_variables")
energy_metadata = spark.table(f"mal_maite_common_{current_environment}.gold.energy_metadata")
batches_production_monitoring = spark.table(f"{source_catalog}.batches_production_monitoring")
manual_entries = spark.table(f"{source_catalog}.manual_entries")
parameters_localizations = spark.table(f"{source_catalog}.parameters_localizations")
requirement_specifications = spark.table(f"{source_catalog}.requirement_specifications")
goods_varieties = spark.table(f"{source_catalog}.goods_varieties")

# localization_events existe en 1 table par site (comme les mesures dans l'ancien projet).
# Les sites ne sont plus listes en dur : on prend ceux declares dans energy_metadata
# (prd_line), qui fait reference pour les sites suivis par le rapport energie. Un
# site ajoute ou retire de energy_metadata est pris en compte au run suivant.
# Un site declare sans table localization_events est ignore, avec alerte, plutot
# que de faire echouer tout le chargement.
sites_energy_metadata = sorted(
    r["prd_line"]
    for r in energy_metadata.select("prd_line").distinct().collect()
    if r["prd_line"]
)

sites = []
for site in sites_energy_metadata:
    table_site = f"mal_maite_{site}_{current_environment}.gold.localization_events"
    if spark.catalog.tableExists(table_site):
        sites.append(site)
    else:
        print(f"ATTENTION : site {site} present dans energy_metadata mais table {table_site} introuvable, site ignore")

if not sites:
    raise ValueError("Aucun site exploitable : energy_metadata ne declare aucun site ayant une table localization_events")
print(f"Sites charges : {sites}")

# Lignes de production (production_line des lots) des sites d'energy_metadata : le
# site (prd_line, ex. rouen1) est le code usine en minuscules de la dimension commune
# dim_site (plant, ex. ROUEN1), celle que lit aussi Power BI. Lots et mesures sont
# limites a ces lignes (dim_batch, fact_batch_measures), pour que le projet ne porte
# que sur les sites d'energy_metadata.
dim_site_commune = spark.table(current_catalog + "." + common_schema + ".dim_site")

lignes_production_suivies = sorted(
    r["id_plant_production_line"]
    for r in dim_site_commune
        .filter(F.lower(F.col("plant")).isin(sites_energy_metadata))
        .select("id_plant_production_line")
        .collect()
)

if not lignes_production_suivies:
    raise ValueError("Aucune ligne de production de dim_site ne correspond aux sites de energy_metadata")
print(f"Lignes de production suivies : {lignes_production_suivies}")

batches_suivis = batches.filter(F.col("production_line").isin(lignes_production_suivies))

# IMPORTANT : select explicite par nom de colonne, pas juste withColumn/table brute.
# Les tables par site sont gerees independamment et peuvent avoir un ordre de
# colonnes different d'un site a l'autre. Comme l'union plus loin (UNION ALL en
# SQL) fonctionne par POSITION et non par nom, un ordre different provoque un
# decalage silencieux des valeurs entre colonnes (ex: batch_id qui se retrouve
# nul car alignee avec une autre colonne vide chez un site donne).
localization_events_by_site = {
    site: spark.table(f"mal_maite_{site}_{current_environment}.gold.localization_events")
          .select(
              "prd_line",
              "prd_workshop",
              "prd_cell",
              "batch_id",
              "localization_event",
              "start",
              "end"
          )
          .withColumn("site", F.lit(site))
    for site in sites
}